# d2oracle

In [ ]:
# 1. Setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ElRizeru/d2oracle.git'
if (Path.cwd() / 'training').exists() and (Path.cwd() / 'shared').exists():
    REPO_ROOT = Path.cwd()
elif Path('/content/d2oracle/training').exists():
    REPO_ROOT = Path('/content/d2oracle')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    REPO_ROOT = Path('/content/d2oracle')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'huggingface-hub>=0.26.0', 'pyarrow>=17.0.0', 'onnx>=1.16.0', 'onnxruntime>=1.19.0', 'matplotlib>=3.9.0',
], check=True)
print('Repo ready:', REPO_ROOT)

In [ ]:
# 2. Config
import logging
import sys
from pathlib import Path

import torch

from training.config import MODEL_SCHEMA_VERSION, TrainingConfig

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| Model schema v:', MODEL_SCHEMA_VERSION)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    stream=sys.stdout,
    force=True,
)

USE_GOOGLE_DRIVE = True
ARTIFACT_ROOT = REPO_ROOT
if USE_GOOGLE_DRIVE and 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_root = Path('/content/drive/MyDrive/d2oracle')
        drive_root.mkdir(parents=True, exist_ok=True)
        ARTIFACT_ROOT = drive_root
    except Exception as exc:
        print('Google Drive unavailable, falling back to local storage:', exc)
DATA_ROOT = Path('/content') if Path('/content').exists() and 'google.colab' in sys.modules else REPO_ROOT
REPORT_DIR = ARTIFACT_ROOT / 'report'
print('Artifact root:', ARTIFACT_ROOT, '| Data root:', DATA_ROOT)

cfg = TrainingConfig(
    hf_repo_id='rizeru/d2oracle-data',
    data_dir=DATA_ROOT / 'data/training',
    checkpoint_dir=ARTIFACT_ROOT / 'checkpoints',
    export_dir=ARTIFACT_ROOT / 'exports',
    d_model=192,
    nhead=6,
    num_encoder_layers=4,
    num_decoder_layers=2,
    dim_feedforward=512,
    dropout=0.1,
    max_item_len=12,
    stage1_pair_dim=24,
    stage1_pair_dropout=0.25,
    use_stage2_block=True,
    eos_logit_penalty=0.0,
    eos_stop_prob=0.48,
    batch_size=256 if torch.cuda.is_available() else 32,
    epochs=25,
    two_phase=True,
    stage1_epochs=8,
    lr=4.5e-4 if torch.cuda.is_available() else 3e-4,
    weight_decay=2e-2,
    val_split=0.1,
    pro_val_min_count=100,
    scenario_teacher_forcing_start=0.5,
    scenario_teacher_forcing_end=0.15,
    duration_teacher_forcing_prob=0.40,
    loser_build_weight=0.35,
    win_label_smoothing=0.02,
    w_win=2.0,
    w_duration=2.0,
    w_item_time=1.5,
    resume=True,
    num_workers=0,
    use_amp=torch.cuda.is_available(),
)
print(cfg)

In [ ]:
# 3. Download dataset
import os

from training.dataset import VocabIndex, download_hf_dataset


def _resolve_hf_token() -> str | None:
    tok = os.environ.get('HF_TOKEN')
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN') or None
    except Exception:
        return None

hf_token = _resolve_hf_token()
manifest, vocab, shard_paths = download_hf_dataset(cfg.hf_repo_id, cfg.data_dir, token=hf_token)
vidx = VocabIndex(vocab)
print(f"Active patch: {manifest['game_version_name']} (gv={manifest['game_version_id']}) | schema={manifest.get('parquet_schema_version')}")
print(f"Shards: {len(shard_paths)} | Declared matches: {manifest.get('total_matches')}")
print(f"Heroes: {vidx.num_heroes} | Item vocab: {vidx.item_vocab_size} | Max hero skills: {vidx.max_hero_skills}")

In [ ]:
# 4. Train
from training.train import train_from_shards

train_result = train_from_shards(shard_paths, vocab, cfg, manifest=manifest)
print('Dataset:', train_result.get('dataset_summary'))
print('Best Stage-1 epoch:', train_result.get('best_stage1_epoch'), '| Best overall epoch:', train_result.get('best_epoch'))
print('Best checkpoint:', train_result['best_checkpoint'])
print('Last val metrics:', train_result['history'][-1]['val'])

In [ ]:
# 5. Plot metrics
from training.diagnostics import plot_training_curves

CURVES_PATH = REPORT_DIR / 'training_curves.png'
plot_training_curves(train_result['history'], out_path=CURVES_PATH, show=True)

In [ ]:
# 6. Export ONNX
from training.export_onnx import export_to_onnx
from training.verify_onnx import verify_onnx_parity

exported = export_to_onnx(train_result['best_checkpoint'], cfg.export_dir)
parity = verify_onnx_parity(train_result['best_checkpoint'], cfg.export_dir, atol=1e-4)

total_mb = 0.0
for name, path in exported.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    total_mb += size_mb
    print(f'{name}: {path} ({size_mb:.2f} MB)')
print(f'Total browser bundle size: {total_mb:.2f} MB')
print('ONNX parity:', parity)

In [ ]:
# 7. Sample prediction
from training.diagnostics import format_sample_comparison

sample_report = format_sample_comparison(
    train_result.get('sample_match'), vidx, vocab, cfg, cfg.export_dir
)
print(sample_report)

In [ ]:
# 8. Package & upload
from huggingface_hub import HfApi

from training.diagnostics import (
    collect_validation_diagnostics,
    format_sample_comparison,
    package_export_bundle,
    plot_training_curves,
    write_training_report,
)

curves_path = REPORT_DIR / 'training_curves.png'
if not curves_path.exists():
    try:
        plot_training_curves(train_result['history'], out_path=curves_path, show=False)
    except Exception as exc:
        print('Skipping training curves plot:', exc)
        curves_path = None

if 'sample_report' not in globals():
    sample_report = format_sample_comparison(
        train_result.get('sample_match'), vidx, vocab, cfg, cfg.export_dir
    )

diagnostics = None
if train_result.get('val_ds') is not None:
    diagnostics = collect_validation_diagnostics(
        train_result['best_checkpoint'],
        train_result['val_ds'],
        vidx,
        cfg,
    )

written_report = write_training_report(
    REPORT_DIR,
    train_result=train_result,
    cfg=cfg,
    vocab=vocab,
    manifest=manifest,
    exported=exported,
    parity=parity,
    diagnostics=diagnostics,
    curves_path=curves_path,
    sample_comparison=sample_report,
)
archive_path = package_export_bundle('d2oracle_exports', cfg.export_dir, REPORT_DIR)
print('Created bundle:', archive_path)
print('Report files:', sorted(p.name for p in written_report.values()))
print('\n' + (REPORT_DIR / 'summary.md').read_text(encoding='utf-8'))

UPLOAD_REPO_ID = ''  # Optional: e.g. 'rizeru/d2oracle-model'
if UPLOAD_REPO_ID:
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or env before uploading.')
    api = HfApi(token=token)
    api.create_repo(repo_id=UPLOAD_REPO_ID, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=str(cfg.export_dir), repo_id=UPLOAD_REPO_ID, repo_type='model')
    print('Uploaded ONNX bundle to', UPLOAD_REPO_ID)